# LungFuseNet — Colab Orchestrator

Run each cell once. Already-completed stages print `[SKIP]` and return instantly.
artifacts/ is symlinked to Google Drive so progress survives runtime resets.

In [ ]:
# Cell 1 — Mount Drive + clone repo + symlink artifacts
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess

REPO_URL = 'https://github.com/YOUR_USERNAME/lung-nodule-fusion-xai.git'
DRIVE_ARTIFACTS = '/content/drive/MyDrive/Kanker Kanker apa yg Kanker/LungFuseNet_artifacts'
REPO_DIR = '/content/lung-nodule-fusion-xai'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

%cd {REPO_DIR}
!pip install -q -r requirements.txt

os.makedirs(DRIVE_ARTIFACTS, exist_ok=True)
if not os.path.exists('artifacts'):
    os.symlink(DRIVE_ARTIFACTS, 'artifacts')
print('Setup done. artifacts ->', os.readlink('artifacts'))

In [ ]:
# Cell 2 — Stage 00: DICOM -> patches + labels.csv  (runs once, skip on resume)
!python src/stage_00_preprocess.py --config configs/config.yaml

In [ ]:
# Cell 3 — Stage 01: patches -> radiomics parquet  (runs once, skip on resume)
!python src/stage_01_radiomics.py --config configs/config.yaml

In [ ]:
# Cell 4 — Stage 02: labels.csv -> folds.json
!python src/stage_02_split.py --config configs/config.yaml

In [ ]:
# Cell 5 — Stage 03: training  (one model at a time; already-done folds skip)
# Change model name between sessions to train the next model.
model = 'mobilenetv3_small'   # options: efficientnet_b0, densenet121, resnet50, vgg16, vit_base

for fold in range(5):
    !python src/stage_03_train.py --config configs/config.yaml --model {model} --fold {fold}

In [ ]:
# Cell 6 — Stage 04: load best checkpoints -> summary.csv
!python src/stage_04_evaluate.py --config configs/config.yaml

In [ ]:
# Cell 7 — Stage 05: Grad-CAM PNGs for Track 1 backbone
!python src/stage_05_xai.py --config configs/config.yaml